# Steam Games Analysis — Executive Summary

A data-driven look at **150,000+ games** on the world's largest PC gaming platform.

This dashboard consolidates key findings from five deep-dive notebooks:
1. **Exploratory Data Analysis** — dataset overview, distributions, and missing data
2. **Genre Trend Predictions** — popularity scoring and 12-month forecasts
3. **Price Sweet-Spot Analysis** — optimal pricing by genre
4. **Review Sentiment Analysis** — NLP on 100K player reviews
5. **Indie vs AAA Comparison** — publisher tier analysis across every dimension

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import (
    load_applications, load_genres, load_application_genres,
    build_games_with_genres, build_games_with_publishers,
    STEAM_COLORS, CHART_COLORS
)
from src.utils import get_plotly_layout

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load core data
games = load_applications()
games_genres = build_games_with_genres(games)
games_pubs = build_games_with_publishers(games)

print("Data loaded!")

---
## Key Metrics at a Glance

In [ ]:
# Compute headline stats
total_games = len(games)
free_games = games['is_free'].sum()
paid_games = total_games - free_games
median_price = games[games['price_dollars'] > 0]['price_dollars'].median()
num_genres = games_genres['genre_name'].nunique()
metacritic_count = (games['metacritic_score'] > 0).sum()
peak_year = int(games['release_year'].dropna().value_counts().idxmax())
peak_count = int(games['release_year'].dropna().value_counts().max())

# Build KPI indicator cards
fig = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'indicator'}] * 4, [{'type': 'indicator'}] * 4],
    vertical_spacing=0.15,
    horizontal_spacing=0.05,
)

kpis = [
    ('Total Games', f'{total_games:,}', 1, 1),
    ('Paid Games', f'{paid_games:,} ({paid_games/total_games*100:.0f}%)', 1, 2),
    ('Free Games', f'{free_games:,} ({free_games/total_games*100:.0f}%)', 1, 3),
    ('Median Price', f'${median_price:.2f}', 1, 4),
    ('Genres', f'{num_genres}', 2, 1),
    ('Peak Release Year', f'{peak_year}', 2, 2),
    ('Games That Year', f'{peak_count:,}', 2, 3),
    ('Have Metacritic', f'{metacritic_count:,} ({metacritic_count/total_games*100:.1f}%)', 2, 4),
]

for title, value, row, col in kpis:
    fig.add_trace(go.Indicator(
        mode='number',
        value=None,
        title=dict(text=f'<b>{title}</b><br><span style="font-size:1.6em;color:#66c0f4">{value}</span>', font=dict(size=14, color='#c7d5e0')),
    ), row=row, col=col)

fig.update_layout(
    paper_bgcolor='#1b2838',
    height=280,
    margin=dict(l=20, r=20, t=30, b=10),
)
fig.show()

---
## 1. Steam's Growth: Games Released Per Year

In [ ]:
# Games released per year (2005-2025)
yearly = (
    games['release_year'].dropna().astype(int)
    .value_counts().sort_index()
)
yearly = yearly[(yearly.index >= 2005) & (yearly.index <= 2025)]

# Color the peak year differently
colors = [STEAM_COLORS['accent_orange'] if y == yearly.idxmax() else STEAM_COLORS['light_blue']
          for y in yearly.index]

fig = go.Figure(go.Bar(
    x=yearly.index.astype(str), y=yearly.values,
    marker_color=colors,
    hovertemplate='%{x}: %{y:,} games<extra></extra>',
))
fig.update_layout(get_plotly_layout(
    'Games Released Per Year on Steam', 'Year', 'Games Released'))
fig.update_layout(height=380)
fig.show()

print(f"Steam's catalog has exploded — {peak_year} saw {peak_count:,} game releases, "
      f"up from just {yearly.iloc[0]:,} in {yearly.index[0]}.")

---
## 2. Genre Landscape: What Developers Are Building

In [ ]:
# Top 15 genres by game count
genre_counts = (
    games_genres.groupby('genre_name')['appid'].nunique()
    .sort_values(ascending=False).head(15)
)

fig = go.Figure(go.Treemap(
    labels=genre_counts.index.tolist(),
    parents=[''] * len(genre_counts),
    values=genre_counts.values.tolist(),
    textinfo='label+value',
    texttemplate='<b>%{label}</b><br>%{value:,} games',
    marker=dict(
        colors=genre_counts.values,
        colorscale=[[0, '#2a475e'], [1, '#66c0f4']],
        line=dict(width=2, color='#1b2838'),
    ),
    hovertemplate='<b>%{label}</b><br>%{value:,} games<extra></extra>',
))
fig.update_layout(
    title=dict(text='Top 15 Genres by Game Count', font=dict(size=20, color='#c7d5e0'), x=0.5),
    paper_bgcolor='#1b2838',
    margin=dict(l=10, r=10, t=50, b=10),
    height=420,
)
fig.show()

print(f"Indie ({genre_counts.iloc[0]:,}) and Action ({genre_counts.iloc[1]:,}) dominate, "
      f"reflecting Steam's accessibility to small developers.")

---
## 3. Price Sweet Spots by Genre

In [ ]:
# Price bracket analysis for top genres
paid = games_genres[(games_genres['is_free'] == False) & (games_genres['price_dollars'] > 0)].copy()
bracket_bins = [0, 5, 10, 15, 20, 30, 50, 100, float('inf')]
bracket_labels = ['$0-5', '$5-10', '$10-15', '$15-20', '$20-30', '$30-50', '$50-100', '$100+']
paid['price_bracket'] = pd.cut(paid['price_dollars'], bins=bracket_bins, labels=bracket_labels, right=True)

# Get top 8 English genres
english_genres = ['Indie', 'Action', 'Adventure', 'Casual', 'Simulation', 'Strategy', 'RPG', 'Sports']
top_paid = paid[paid['genre_name'].isin(english_genres)]

# Find sweet spot per genre (bracket with highest avg recommendations, min 10 games)
genre_bracket = (
    top_paid.groupby(['genre_name', 'price_bracket'], observed=False)
    .agg(avg_recs=('recommendations_total', 'mean'), game_count=('appid', 'nunique'))
    .reset_index()
)
genre_bracket = genre_bracket[genre_bracket['game_count'] >= 10]
best = genre_bracket.loc[genre_bracket.groupby('genre_name')['avg_recs'].idxmax()]
best = best.sort_values('avg_recs', ascending=True)

# Horizontal bar chart showing sweet spot bracket and avg recs
fig = go.Figure()
fig.add_trace(go.Bar(
    y=[f"{row['genre_name']}  ({row['price_bracket']})" for _, row in best.iterrows()],
    x=best['avg_recs'].values,
    orientation='h',
    marker_color=STEAM_COLORS['light_blue'],
    text=[f"{v:,.0f} avg recs" for v in best['avg_recs']],
    textposition='outside',
    textfont=dict(color='#c7d5e0'),
    hovertemplate='<b>%{y}</b><br>Avg Recommendations: %{x:,.0f}<extra></extra>',
))
fig.update_layout(get_plotly_layout(
    'Price Sweet Spot by Genre (Best Bracket for Recommendations)',
    xaxis_title='Average Recommendations', yaxis_title=''))
fig.update_layout(height=400, margin=dict(l=200))
fig.show()

print("Each bar shows the price bracket where games in that genre get the most recommendations.")
print("Price alone has almost no correlation with success (r = 0.035).")

---
## 4. Player Sentiment: What Reviews Tell Us

In [ ]:
# Pre-computed sentiment stats from NB04 (avoids re-running TextBlob)
# These values come from our 100K sample, 53K English reviews
sentiment_data = {
    'Category': ['Positive\n(> 0.05)', 'Neutral\n(-0.05 to 0.05)', 'Negative\n(< -0.05)'],
    'Count': [29526, 13377, 10118],
    'Color': [STEAM_COLORS['accent_green'], STEAM_COLORS['accent_orange'], STEAM_COLORS['accent_red']],
}

thumbs_data = {
    'Vote': ['Thumbs Up', 'Thumbs Down'],
    'Avg Sentiment': [0.156, -0.065],
    'Color': [STEAM_COLORS['accent_green'], STEAM_COLORS['accent_red']],
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Sentiment Distribution (53K English Reviews)', 'TextBlob vs Thumbs Up/Down'],
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    horizontal_spacing=0.12,
)

# Pie chart — sentiment breakdown
fig.add_trace(go.Pie(
    labels=sentiment_data['Category'],
    values=sentiment_data['Count'],
    marker=dict(colors=sentiment_data['Color'], line=dict(color='#1b2838', width=2)),
    textinfo='label+percent',
    textfont=dict(size=12),
    hovertemplate='%{label}<br>%{value:,} reviews<extra></extra>',
), row=1, col=1)

# Bar chart — avg sentiment by vote type
fig.add_trace(go.Bar(
    x=thumbs_data['Vote'],
    y=thumbs_data['Avg Sentiment'],
    marker_color=thumbs_data['Color'],
    text=[f'{v:.3f}' for v in thumbs_data['Avg Sentiment']],
    textposition='outside',
    textfont=dict(color='#c7d5e0', size=16),
    hovertemplate='<b>%{x}</b><br>Avg Polarity: %{y:.3f}<extra></extra>',
    showlegend=False,
), row=1, col=2)

fig.update_layout(
    paper_bgcolor='#1b2838', plot_bgcolor='#2a475e',
    font=dict(color='#c7d5e0', size=12),
    height=380, showlegend=False,
    margin=dict(t=50, b=40),
)
for ann in fig['layout']['annotations']:
    ann['font'] = dict(size=14, color='#c7d5e0')
fig.update_yaxes(gridcolor='rgba(198,213,224,0.15)', row=1, col=2)
fig.show()

print("55.7% of reviews are positive. TextBlob agrees with thumbs up/down 71.9% of the time.")
print("Sentiment is stable over time (~0.10 avg polarity from 2015-2025).")

---
## 5. Indie vs AAA: Who Wins?

In [ ]:
# Classify publishers into tiers
pub_counts = games_pubs.groupby('publisher_name')['appid'].nunique().reset_index()
pub_counts.columns = ['publisher_name', 'game_count']
pub_counts['tier'] = pub_counts['game_count'].apply(
    lambda x: 'AAA' if x >= 50 else ('Mid-tier' if x >= 6 else 'Indie'))
games_tier = games_pubs.merge(pub_counts[['publisher_name', 'tier']], on='publisher_name', how='left')
games_tier_dedup = games_tier.drop_duplicates(subset='appid')

tier_order = ['Indie', 'Mid-tier', 'AAA']
tier_colors = {'Indie': '#66c0f4', 'Mid-tier': '#ff9800', 'AAA': '#4caf50'}

# Compute comparison metrics
comparison = []
for tier in tier_order:
    td = games_tier_dedup[games_tier_dedup['tier'] == tier]
    tm = td[td['metacritic_score'] > 0]
    comparison.append({
        'Tier': tier,
        'Games': td['appid'].nunique(),
        'Median Price': td['price_dollars'].median(),
        'F2P %': td['is_free'].mean() * 100,
        'Avg Metacritic': tm['metacritic_score'].mean() if len(tm) > 0 else 0,
        'Median Recs': td['recommendations_total'].median(),
    })
comp_df = pd.DataFrame(comparison)

# Multi-metric comparison chart
metrics = [
    ('Median Price ($)', 'Median Price', lambda v: f'${v:.2f}'),
    ('F2P Rate (%)', 'F2P %', lambda v: f'{v:.1f}%'),
    ('Avg Metacritic', 'Avg Metacritic', lambda v: f'{v:.1f}'),
    ('Median Recommendations', 'Median Recs', lambda v: f'{v:,.0f}'),
]

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[m[0] for m in metrics],
    horizontal_spacing=0.08,
)

for col_idx, (title, col_name, fmt_fn) in enumerate(metrics, 1):
    for tier in tier_order:
        val = comp_df[comp_df['Tier'] == tier][col_name].values[0]
        fig.add_trace(go.Bar(
            x=[tier], y=[val],
            marker_color=tier_colors[tier],
            text=[fmt_fn(val)], textposition='outside',
            textfont=dict(color='#c7d5e0', size=11),
            showlegend=(col_idx == 1), name=tier,
            legendgroup=tier,
        ), row=1, col=col_idx)

fig.update_layout(
    paper_bgcolor='#1b2838', plot_bgcolor='#2a475e',
    font=dict(color='#c7d5e0', size=11),
    height=380, barmode='group',
    legend=dict(bgcolor='rgba(42,71,94,0.8)', bordercolor='#c7d5e0', borderwidth=1,
               orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5),
    margin=dict(t=80, b=40),
)
for ann in fig['layout']['annotations']:
    ann['font'] = dict(size=12, color='#c7d5e0')
fig.update_yaxes(gridcolor='rgba(198,213,224,0.15)')
fig.update_xaxes(showticklabels=False)
fig.show()

print("Metacritic scores are nearly identical across tiers (72.6 / 74.5 / 74.2) — "
      "indie games absolutely compete on quality.")
print("Indie releases grew +214.7% (2020-25 vs 2015-19), outpacing AAA at +107.9%.")

---
## 6. The Most Recommended Games on Steam

In [ ]:
# Top 15 most recommended games
top_games = (
    games[['name', 'recommendations_total', 'price_dollars', 'metacritic_score']]
    .dropna(subset=['recommendations_total'])
    .sort_values('recommendations_total', ascending=True)
    .tail(15)
)

# Color by free vs paid
bar_colors = [STEAM_COLORS['accent_green'] if p == 0 else STEAM_COLORS['light_blue']
              for p in top_games['price_dollars']]

fig = go.Figure(go.Bar(
    y=top_games['name'],
    x=top_games['recommendations_total'],
    orientation='h',
    marker_color=bar_colors,
    text=[f"{v:,.0f}" for v in top_games['recommendations_total']],
    textposition='outside',
    textfont=dict(color='#c7d5e0', size=11),
    hovertemplate='<b>%{y}</b><br>Recommendations: %{x:,.0f}<extra></extra>',
))

fig.update_layout(get_plotly_layout(
    'Top 15 Most Recommended Games on Steam',
    xaxis_title='Total Recommendations', yaxis_title=''))
fig.update_layout(height=480, margin=dict(l=250))

# Add legend annotation
fig.add_annotation(
    x=0.98, y=0.02, xref='paper', yref='paper',
    text='<span style="color:#4caf50">■</span> Free  <span style="color:#66c0f4">■</span> Paid',
    showarrow=False, font=dict(size=13, color='#c7d5e0'),
    bgcolor='rgba(42,71,94,0.8)', bordercolor='#c7d5e0', borderwidth=1, borderpad=6,
)
fig.show()

---
## Key Takeaways

| Finding | Detail |
|---------|--------|
| **Steam is booming** | 20K+ games released in 2024 alone, up from ~100 in 2005 |
| **Indie dominates the catalog** | 103K indie games vs 14K AAA — and growing 2x faster |
| **Price ≠ success** | Correlation between price and recommendations is essentially zero (r = 0.035) |
| **Quality is tier-agnostic** | Metacritic scores are nearly identical across indie/mid/AAA tiers |
| **Reviews skew positive** | 55.7% of reviews are positive; sentiment has been stable since 2015 |
| **Sweet spots vary by genre** | RPGs and Action do best at $50-100; Indie titles peak at $30-50 |
| **Recommendations are winner-take-all** | Top 5 games have more recs than the bottom 100K combined |

### For Indie Developers
1. **Pick a niche** — find genres where AAA isn't dominating
2. **Price for your genre** — check the sweet-spot data, don't race to the bottom
3. **Quality over budget** — the data proves small teams can compete on review scores
4. **The market is growing** — more competition, but also a larger and more diverse audience

---
*Analysis based on Steam Games Dataset (2025) — 240K applications, 150K games, 4M+ reviews.*